# AnimalEmotion AI – Deep Learning-Based Animal Emotion Recognition System
## Phase 1: Dataset Audit

A reproducible, evidence-driven audit of the Kaggle dataset. This notebook does not train a model, create splits, or modify/delete original images. Folder names are inferred metadata, not confirmed labels.

Dataset: https://www.kaggle.com/datasets/kk2004/animalemotion/data

## Audit plan
Discover dataset contents, safely extract the archive, inspect images and annotation files, verify image integrity, compute exact SHA-256 duplicates, optionally compare perceptual hashes, analyze dimensions/channels/sizes/labels, review imbalance and leakage risks, and export CSV/JSON reports. Findings are based only on downloaded evidence; no class names or counts are assumed.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, hashlib, stat, shutil, random, re, subprocess, datetime
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError, ImageFile
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torchvision, cv2, sklearn

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
sns.set_theme(style='whitegrid')
ROOT = Path('/content/animalemotion'); RAW = ROOT/'raw'; DATA = ROOT/'dataset'; REPORTS = ROOT/'reports'
for folder in (ROOT, RAW, DATA, REPORTS): folder.mkdir(parents=True, exist_ok=True)
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__, '| OpenCV', cv2.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## Kaggle authentication and download
Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab Secrets and allow notebook access. If unavailable, upload your Kaggle `kaggle.json`. Secrets are never hardcoded or printed.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
username = key = None
try:
    from google.colab import userdata
    username, key = userdata.get('KAGGLE_USERNAME'), userdata.get('KAGGLE_KEY')
except Exception:
    pass
kaggle_cfg = Path.home()/'.kaggle'/'kaggle.json'; kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
if not (username and key) and kaggle_cfg.exists():
    try:
        cfg = json.loads(kaggle_cfg.read_text()); username, key = cfg.get('username'), cfg.get('key')
    except Exception as exc: print('Could not read existing token file:', exc)
if not (username and key):
    try:
        from google.colab import files
        print('Upload kaggle.json. Credential contents will not be displayed.')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            kaggle_cfg.write_bytes(uploaded['kaggle.json']); cfg = json.loads(kaggle_cfg.read_text())
            username, key = cfg.get('username'), cfg.get('key')
    except Exception as exc: print('Upload unavailable:', exc)
if not (username and key): raise RuntimeError('Kaggle credentials unavailable. Configure Colab Secrets or upload kaggle.json.')
kaggle_cfg.write_text(json.dumps({'username': username, 'key': key})); os.chmod(kaggle_cfg, 0o600)
os.environ['KAGGLE_USERNAME'] = username; os.environ['KAGGLE_KEY'] = key
SLUG = 'kk2004/animalemotion'
if not list(RAW.glob('*.zip')):
    proc = subprocess.run(['kaggle', 'datasets', 'download', '-d', SLUG, '-p', str(RAW)], capture_output=True, text=True)
    if proc.returncode: raise RuntimeError('Kaggle download failed: ' + proc.stderr[-2000:])
archives = sorted(RAW.glob('*.zip'))
if not archives: raise FileNotFoundError('No ZIP archive found in ' + str(RAW))
print('Archive files:', [p.name for p in archives])

## Safe ZIP extraction
Absolute paths, `..` path components, symbolic links, and resolved paths escaping the extraction root are rejected. The archive is treated as untrusted input.

In [ ]:
def safe_extract(archive: Path, destination: Path):
    root = destination.resolve(); accepted = []; rejected = []
    with zipfile.ZipFile(archive) as zf:
        for info in zf.infolist():
            name = info.filename; rel = Path(name)
            mode = info.external_attr >> 16
            if name.startswith('/') or re.match(r'^[A-Za-z]:', name) or '..' in rel.parts or stat.S_ISLNK(mode):
                rejected.append({'entry': name, 'reason': 'unsafe path or symbolic link'}); continue
            target = (root/rel).resolve()
            if target != root and root not in target.parents:
                rejected.append({'entry': name, 'reason': 'resolved outside extraction root'}); continue
            if info.is_dir(): target.mkdir(parents=True, exist_ok=True); continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, open(target, 'wb') as dst: shutil.copyfileobj(src, dst)
            accepted.append(name)
    return accepted, rejected
accepted, rejected = [], []
for archive in archives:
    a, r = safe_extract(archive, DATA); accepted.extend(a); rejected.extend(r)
(REPORTS/'safe_extraction.json').write_text(json.dumps({'extracted_entries': len(accepted), 'rejected_entries': rejected}, indent=2))
print('Extracted:', len(accepted), '| rejected:', len(rejected))

## Recursive inspection, image discovery, and integrity audit

In [ ]:
IMAGE_EXTS = {'.jpg','.jpeg','.png','.bmp','.gif','.tif','.tiff','.webp','.jfif','.heic','.heif'}
ANNOT_EXTS = {'.csv','.json','.xlsx','.xls','.yaml','.yml','.xml','.txt'}
all_files = [p for p in DATA.rglob('*') if p.is_file()]
image_files = [p for p in all_files if p.suffix.lower() in IMAGE_EXTS]
annotation_files = [p for p in all_files if p.suffix.lower() in ANNOT_EXTS]
print(f'{len(all_files)} total files; {len(image_files)} image candidates; {len(annotation_files)} annotation candidates')
folder_counts = {}
for p in all_files:
    rel = p.relative_to(DATA)
    for i in range(1, len(rel.parts)):
        name = '/'.join(rel.parts[:i]); folder_counts[name] = folder_counts.get(name, 0) + 1
pd.DataFrame([{'folder': k, 'file_count': v} for k,v in sorted(folder_counts.items())]).to_csv(REPORTS/'folder_hierarchy.csv', index=False)
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''): digest.update(block)
    return digest.hexdigest()
ImageFile.LOAD_TRUNCATED_IMAGES = False
rows = []
for path in tqdm(image_files, desc='Image audit'):
    rel = path.relative_to(DATA).as_posix(); base = {'relative_path': rel, 'filename': path.name, 'file_bytes': path.stat().st_size, 'sha256': None}
    try:
        base['sha256'] = sha256_file(path)
        with Image.open(path) as im: im.verify()
        with Image.open(path) as im:
            im.load(); width, height = im.size; mode = im.mode
            base.update({'integrity':'ok', 'format':im.format, 'mode':mode, 'channels':len(im.getbands()), 'width':width, 'height':height, 'aspect_ratio':width/height if height else None, 'has_transparency':('A' in im.getbands() or 'transparency' in im.info), 'error':''})
    except Exception as exc:
        base.update({'integrity':'corrupted_or_unreadable', 'format':None, 'mode':None, 'channels':None, 'width':None, 'height':None, 'aspect_ratio':None, 'has_transparency':None, 'error':f'{type(exc).__name__}: {exc}'})
    rows.append(base)
inventory = pd.DataFrame(rows)
inventory.to_csv(REPORTS/'image_inventory.csv', index=False)
corrupt = inventory[inventory.integrity != 'ok'].copy()
corrupt.to_csv(REPORTS/'corrupted_images.csv', index=False)
valid = inventory[inventory.integrity == 'ok'].copy()
print('Readable:', len(valid), '| unreadable:', len(corrupt))

## Image properties, duplicate detection, and visual review

In [ ]:
if not valid.empty:
    dimensions = {'readable_images':len(valid), 'width_min':int(valid.width.min()), 'width_median':float(valid.width.median()), 'width_max':int(valid.width.max()), 'height_min':int(valid.height.min()), 'height_median':float(valid.height.median()), 'height_max':int(valid.height.max()), 'aspect_ratio_median':float(valid.aspect_ratio.median()), 'formats':valid['format'].value_counts().to_dict(), 'modes':valid['mode'].value_counts().to_dict(), 'transparent_count':int(valid.has_transparency.sum()), 'file_bytes_min':int(valid.file_bytes.min()), 'file_bytes_median':float(valid.file_bytes.median()), 'file_bytes_max':int(valid.file_bytes.max())}
else: dimensions = {'readable_images':0}
(REPORTS/'dimension_summary.json').write_text(json.dumps(dimensions, indent=2))
dup_hashes = valid.groupby('sha256').filter(lambda group: len(group)>1).sort_values(['sha256','relative_path']) if not valid.empty else valid
dup_hashes.to_csv(REPORTS/'exact_duplicates.csv', index=False)
dup_groups = dup_hashes.groupby('sha256').agg(paths=('relative_path', lambda values: ' | '.join(values)), count=('relative_path','size')).reset_index() if not dup_hashes.empty else pd.DataFrame(columns=['sha256','paths','count'])
dup_groups.to_csv(REPORTS/'exact_duplicate_groups.csv', index=False)
ENABLE_PHASH = False
if ENABLE_PHASH:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ImageHash'], check=True)
    import imagehash
    hashes=[]
    for rel in tqdm(valid.relative_path, desc='Perceptual hashes'):
        with Image.open(DATA/rel) as im: hashes.append({'relative_path':rel, 'phash':str(imagehash.phash(ImageOps.exif_transpose(im).convert('RGB'), hash_size=16))})
    phdf=pd.DataFrame(hashes); phdf.to_csv(REPORTS/'perceptual_hashes.csv', index=False)
    pairs=[]
    for i, a in enumerate(hashes):
        for b in hashes[i+1:]:
            distance=imagehash.hex_to_hash(a['phash'])-imagehash.hex_to_hash(b['phash'])
            if distance <= 6: pairs.append({'path_a':a['relative_path'],'path_b':b['relative_path'],'hamming_distance':int(distance)})
    pd.DataFrame(pairs).to_csv(REPORTS/'perceptual_duplicate_candidates.csv', index=False)
else: pd.DataFrame(columns=['relative_path','phash']).to_csv(REPORTS/'perceptual_hashes.csv', index=False)
if not valid.empty:
    fig, axes = plt.subplots(2,2,figsize=(13,9))
    axes[0,0].hist(valid.width, bins=40); axes[0,0].set_title('Width (pixels)')
    axes[0,1].hist(valid.height, bins=40); axes[0,1].set_title('Height (pixels)')
    axes[1,0].scatter(valid.width, valid.height, alpha=.4); axes[1,0].set_title('Resolution')
    axes[1,1].hist(valid.file_bytes/1024, bins=40); axes[1,1].set_title('File size (KB)')
    plt.tight_layout(); plt.savefig(REPORTS/'image_property_plots.png', dpi=150); plt.show()
def show_mosaic(paths, title, filename, columns=5):
    paths=list(paths)[:20]
    if not paths: print(title + ': no examples'); return
    fig, axes=plt.subplots((len(paths)+columns-1)//columns, columns, figsize=(columns*3, ((len(paths)+columns-1)//columns)*3)); axes=np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis('off')
    for ax, rel in zip(axes, paths):
        try:
            with Image.open(DATA/rel) as im: ax.imshow(im.convert('RGB')); ax.set_title(Path(rel).name[:25], fontsize=8)
        except Exception: ax.text(.5,.5,'Unreadable',ha='center')
    fig.suptitle(title); plt.tight_layout(); plt.savefig(REPORTS/filename, dpi=140); plt.show()
rng=np.random.default_rng(SEED)
sample=valid.relative_path.tolist(); rng.shuffle(sample)
show_mosaic(sample, 'Random readable image examples', 'sample_images.png')
show_mosaic(dup_hashes.relative_path.drop_duplicates().tolist(), 'Exact duplicate examples', 'duplicate_examples.png')

## Annotation inspection and evidence-based label discovery
CSV columns with explicit emotion/species-style names are candidate annotation evidence, not proof of correct image mapping. Generic columns require manual review. JSON key structures are inventoried. Folder components are always reported as inferred only.

In [ ]:
annotation_manifest=[]; csv_tables={}; json_data={}
for path in annotation_files:
    item={'relative_path':path.relative_to(DATA).as_posix(), 'suffix':path.suffix.lower(), 'status':'unreadable', 'row_count':None, 'columns':[], 'keys':[], 'error':''}
    try:
        if path.suffix.lower()=='.csv':
            table=pd.read_csv(path); csv_tables[item['relative_path']]=table
            item.update(status='readable', row_count=len(table), columns=[str(c) for c in table.columns])
        elif path.suffix.lower()=='.json':
            obj=json.loads(path.read_text(encoding='utf-8')); json_data[item['relative_path']]=obj
            item.update(status='readable', row_count=len(obj) if isinstance(obj,list) else None, keys=list(obj.keys())[:100] if isinstance(obj,dict) else (list(obj[0].keys())[:100] if obj and isinstance(obj[0],dict) else []))
        elif path.suffix.lower() in {'.xlsx','.xls'}:
            book=pd.ExcelFile(path); item.update(status='readable', columns=book.sheet_names)
        else: item.update(status='present_not_parsed')
    except Exception as exc: item['error']=f'{type(exc).__name__}: {exc}'
    annotation_manifest.append(item)
pd.DataFrame(annotation_manifest).to_csv(REPORTS/'annotation_manifest.csv', index=False)
emotion_names={'emotion','emotions','expression','expressions','sentiment','mood'}; species_names={'species','animal','animal_type','animal_class','breed'}; generic_names={'label','class','category','target'}
def normalize_name(value): return re.sub('[^a-z0-9]+','_',str(value).lower()).strip('_')
evidence=[]; distributions=[]
for rel, table in csv_tables.items():
    for column in table.columns:
        name=normalize_name(column)
        kind='confirmed_emotion_column' if name in emotion_names else ('confirmed_species_column' if name in species_names else ('candidate_label_requires_review' if name in generic_names else None))
        if kind:
            counts=table[column].fillna('<missing>').astype(str).value_counts()
            evidence.append({'source':rel,'column':str(column),'evidence_type':kind,'unique_values':int(table[column].nunique(dropna=True)),'top_values':counts.head(30).to_dict()})
            distributions.extend([{'source':rel,'column':str(column),'evidence_type':kind,'label':str(label),'count':int(count)} for label,count in counts.items()])
pd.DataFrame(evidence).to_csv(REPORTS/'label_evidence.csv', index=False)
class_dist=pd.DataFrame(distributions, columns=['source','column','evidence_type','label','count'])
class_dist.to_csv(REPORTS/'class_distribution.csv', index=False)
folder_labels=[]
for path in image_files:
    for depth, name in enumerate(path.relative_to(DATA).parts[:-1], 1): folder_labels.append({'image':path.relative_to(DATA).as_posix(),'depth':depth,'folder_name':name,'status':'inferred_folder_name_only','confirmed_label':False})
pd.DataFrame(folder_labels).to_csv(REPORTS/'inferred_folder_names.csv', index=False)
imbalance=[]
for (source,column), group in class_dist.groupby(['source','column']):
    counts=group['count']; imbalance.append({'source':source,'column':column,'class_count':len(counts),'total_rows':int(counts.sum()),'min_class':int(counts.min()),'max_class':int(counts.max()),'max_min_ratio':float(counts.max()/counts.min()) if counts.min() else None})
pd.DataFrame(imbalance).to_csv(REPORTS/'imbalance_analysis.csv', index=False)
if not class_dist.empty:
    plt.figure(figsize=(12,max(5,min(16,len(class_dist)/2)))); sns.barplot(data=class_dist.sort_values('count',ascending=False),x='count',y='label',hue='column'); plt.tight_layout(); plt.savefig(REPORTS/'class_distribution.png',dpi=150); plt.show()
print('Annotation files:',len(annotation_manifest),'| label evidence rows:',len(evidence))
if not evidence: print('No explicit label columns found. Folder names are not treated as confirmed labels.')

## Suspicious records, leakage assessment, and audit verdict

In [ ]:
suspicious=[]
for _, row in class_dist[class_dist['count'] < 5].iterrows(): suspicious.append({'issue':'rare_label_candidate','source':row['source'],'column':row['column'],'label':row['label'],'count':int(row['count'])})
stem_counts=valid.assign(stem=valid.filename.map(lambda value:Path(value).stem)).groupby('stem').size() if not valid.empty else pd.Series(dtype=int)
for stem,count in stem_counts[stem_counts>1].items(): suspicious.append({'issue':'duplicate_filename_stem','stem':stem,'count':int(count)})
if not valid.empty:
    for _, row in valid[(valid.width<32)|(valid.height<32)|(valid.aspect_ratio<0.25)|(valid.aspect_ratio>4)].iterrows(): suspicious.append({'issue':'unusual_dimensions_or_aspect_ratio','path':row['relative_path'],'width':int(row['width']),'height':int(row['height']),'aspect_ratio':float(row['aspect_ratio'])})
pd.DataFrame(suspicious).to_csv(REPORTS/'suspicious_review.csv', index=False)
issues=[]
if len(corrupt): issues.append(f'{len(corrupt)} unreadable image candidates')
if len(dup_groups): issues.append(f'{len(dup_groups)} exact SHA-256 duplicate groups')
if not evidence: issues.append('No explicit emotion/species annotation columns confirmed')
leakage={'exact_duplicate_groups':int(len(dup_groups)),'files_in_exact_duplicate_groups':int(len(dup_hashes)),'duplicate_filename_stems':int((stem_counts>1).sum()),'perceptual_hashing_enabled':ENABLE_PHASH,'split_created':False,'assessment':'Potential leakage risk remains; reconcile exact and optional perceptual duplicates before any future split.'}
(REPORTS/'leakage_assessment.json').write_text(json.dumps(leakage,indent=2))
suitability='potentially suitable, pending human verification of annotation mapping and duplicates' if evidence and not len(corrupt) else ('not yet suitable for supervised training: no explicit annotation evidence' if not evidence else 'requires review: unreadable image candidates detected')
verdict={'phase':1,'dataset_suitability':suitability,'file_counts':{'total_files':len(all_files),'image_candidates':len(image_files),'readable_images':len(valid),'unreadable_images':len(corrupt),'annotation_candidates':len(annotation_files)},'detected_issues':issues,'known_limitations':['Folder names are inferred, not confirmed labels','CSV column names do not prove image-to-label mapping','JSON structures are inventoried but not automatically interpreted as labels','Perceptual duplicate audit is optional','No training or data split performed'],'unresolved_questions':['What is the authoritative image-to-label mapping?','Are species and emotion separate targets?','Are folder names canonical labels?','Do near-duplicates occur across potential future splits?'],'phase_2_recommendations':['Manually validate annotation mapping on sampled records','Document label ontology and hierarchy','Review corrupt files and duplicate groups without modifying originals','Run perceptual hash review if scale permits','Only then design a reproducible leakage-aware split']}
(REPORTS/'phase1_verdict.json').write_text(json.dumps(verdict,indent=2))
summary={'project':'AnimalEmotion AI','dataset_slug':SLUG,'seed':SEED,'verdict':verdict,'leakage_assessment':leakage,'image_properties':dimensions,'label_evidence':evidence}
(REPORTS/'audit_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(verdict,indent=2))

## Reports and download
All generated reports are written under `/content/animalemotion/reports`. The ZIP bundle can be downloaded from Colab. No source image is automatically removed or modified.

In [ ]:
manifest={'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'report_files':sorted(p.name for p in REPORTS.iterdir() if p.is_file())}
(REPORTS/'REPORT_MANIFEST.json').write_text(json.dumps(manifest,indent=2))
required=['image_inventory.csv','corrupted_images.csv','folder_hierarchy.csv','annotation_manifest.csv','audit_summary.json','phase1_verdict.json']
missing=[name for name in required if not (REPORTS/name).exists()]
if missing: raise RuntimeError('Required reports missing: '+', '.join(missing))
bundle=ROOT/'animalemotion_phase1_audit_reports.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(REPORTS.iterdir()):
        if path.is_file(): archive.write(path,arcname='reports/'+path.name)
print('Audit reports:',REPORTS,'| bundle:',bundle)
try:
    from google.colab import files
    files.download(str(bundle))
except Exception as exc: print('Automatic browser download is available only in Colab:',exc)
print('Notebook audit completed; no model training or data split was performed.')